#### ─── Notebook 04: Data Quality Checks ───────────────────────────
#### Run this AFTER notebooks 01, 02, 03 are complete
#### All tables read from Unity Catalog — no file paths needed

In [0]:

BRONZE_TABLE  = "workspace.new_project.bronze_online_retail"
SILVER_TABLE  = "workspace.new_project.silver_online_retail"
GOLD_COUNTRY  = "workspace.new_project.gold_revenue_by_country"
GOLD_PRODUCTS = "workspace.new_project.gold_top_products"
GOLD_MONTHLY  = "workspace.new_project.gold_monthly_trend"
GOLD_RFM      = "workspace.new_project.gold_customer_rfm"

# Track overall results
results = []

def check(name, passed, detail=""):
    status = "✅ PASS" if passed else "❌ FAIL"
    results.append({"check": name, "status": status, "detail": detail})
    print(f"{status} | {name} | {detail}")

print("=" * 60)
print("  DATA QUALITY REPORT")
print("=" * 60)

  DATA QUALITY REPORT


#### Check 1: Row Count — Bronze vs Silver


In [0]:
from pyspark.sql.functions import col, count, when, countDistinct
from pyspark.sql.functions import min, max, month, year

bronze = spark.read.table(BRONZE_TABLE)
silver = spark.read.table(SILVER_TABLE)

bronze_count = bronze.count()
silver_count = silver.count()
removed      = bronze_count - silver_count
pct_removed  = round((removed / bronze_count) * 100, 1)

print(f"\nBronze rows : {bronze_count:,}")
print(f"Silver rows : {silver_count:,}")
print(f"Removed     : {removed:,} rows ({pct_removed}%)\n")

# Silver should have fewer rows than Bronze (we cleaned data)
check("Silver < Bronze",        silver_count < bronze_count,
      f"{silver_count:,} < {bronze_count:,}")

# Silver should keep at least 60% of Bronze rows
check("Silver retains >= 60%",  pct_removed <= 40,
      f"Removed {pct_removed}% of rows")

# Silver must have meaningful rows
check("Silver has 100K+ rows",  silver_count >= 100_000,
      f"{silver_count:,} rows")


Bronze rows : 1,067,371
Silver rows : 768,862
Removed     : 298,509 rows (28.0%)

✅ PASS | Silver < Bronze | 768,862 < 1,067,371
✅ PASS | Silver retains >= 60% | Removed 28.0% of rows
✅ PASS | Silver has 100K+ rows | 768,862 rows


#### ── Check 2: Null Check on Critical Columns ──────────────────────

In [0]:
critical_cols = ["invoice_id", "customer_id", "stock_code",
                 "quantity", "unit_price", "total_amount"]

null_counts = silver.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in critical_cols
]).collect()[0]

print("\nNull counts per column:")
for c in critical_cols:
    nulls = null_counts[c]
    print(f"  {c:<20} : {nulls:,} nulls")
    check(f"No nulls in {c}", nulls == 0, f"{nulls} nulls found")


Null counts per column:
  invoice_id           : 0 nulls
✅ PASS | No nulls in invoice_id | 0 nulls found
  customer_id          : 0 nulls
✅ PASS | No nulls in customer_id | 0 nulls found
  stock_code           : 0 nulls
✅ PASS | No nulls in stock_code | 0 nulls found
  quantity             : 0 nulls
✅ PASS | No nulls in quantity | 0 nulls found
  unit_price           : 0 nulls
✅ PASS | No nulls in unit_price | 0 nulls found
  total_amount         : 0 nulls
✅ PASS | No nulls in total_amount | 0 nulls found


#### ── Check 3: Duplicate Check

In [0]:
total_rows  = silver.count()
unique_rows = silver.dropDuplicates(["invoice_id", "stock_code"]).count()
dupes       = total_rows - unique_rows

check("No duplicate invoice+stock rows",
      dupes == 0,
      f"{dupes} duplicate rows found")

✅ PASS | No duplicate invoice+stock rows | 0 duplicate rows found


#### ── Check 4: Business Rule Validations

In [0]:
# No negative or zero quantities
neg_qty = silver.filter(col("quantity") <= 0).count()
check("All quantities > 0", neg_qty == 0,
      f"{neg_qty} rows with qty <= 0")

# No negative or zero prices
neg_price = silver.filter(col("unit_price") <= 0).count()
check("All unit prices > 0", neg_price == 0,
      f"{neg_price} rows with price <= 0")

# No cancellation invoices (should start with C)
cancel_rows = silver.filter(col("invoice_id").startswith("C")).count()
check("No cancellation invoices", cancel_rows == 0,
      f"{cancel_rows} cancellation rows found")

# No negative total amounts
neg_total = silver.filter(col("total_amount") <= 0).count()
check("All total_amount > 0", neg_total == 0,
      f"{neg_total} rows with total_amount <= 0")

# total_amount must equal quantity * unit_price
from pyspark.sql.functions import abs as spark_abs, round as spark_round
mismatch = silver.filter(
    spark_abs(col("total_amount") -
             (col("quantity") * col("unit_price"))) > 0.01
).count()
check("total_amount = qty * price", mismatch == 0,
      f"{mismatch} rows with calculation mismatch")

✅ PASS | All quantities > 0 | 0 rows with qty <= 0
✅ PASS | All unit prices > 0 | 0 rows with price <= 0
✅ PASS | No cancellation invoices | 0 cancellation rows found
✅ PASS | All total_amount > 0 | 0 rows with total_amount <= 0
✅ PASS | total_amount = qty * price | 0 rows with calculation mismatch


####── Check 5: Date Range Validation

In [0]:
date_stats = silver.agg(
    min("invoice_date").alias("min_date"),
    max("invoice_date").alias("max_date")
).collect()[0]

min_date = date_stats["min_date"]
max_date = date_stats["max_date"]

print(f"\nDate range: {min_date} → {max_date}")

# Dates must not be null
check("min_date is not null", min_date is not None, str(min_date))
check("max_date is not null", max_date is not None, str(max_date))

# max date must be after min date
if min_date and max_date:
    check("max_date > min_date", max_date > min_date,
          f"{min_date} to {max_date}")

# Dataset should span at least 12 months
if min_date and max_date:
    from pyspark.sql.functions import datediff, lit
    days_diff = (max_date - min_date).days
    check("Date span >= 12 months", days_diff >= 365,
          f"{days_diff} days span")


Date range: 2009-12-01 07:45:00 → 2011-12-09 12:50:00
✅ PASS | min_date is not null | 2009-12-01 07:45:00
✅ PASS | max_date is not null | 2011-12-09 12:50:00
✅ PASS | max_date > min_date | 2009-12-01 07:45:00 to 2011-12-09 12:50:00
✅ PASS | Date span >= 12 months | 738 days span


#### ── Check 6: Gold Table Validations

In [0]:
# Revenue by country
country_df = spark.read.table(GOLD_COUNTRY)
check("gold_revenue_by_country has rows",
      country_df.count() > 0,
      f"{country_df.count()} countries")

check("gold_revenue_by_country has UK",
      country_df.filter(col("country") == "UNITED KINGDOM").count() > 0,
      "UK is top market in dataset")

# Top products
products_df = spark.read.table(GOLD_PRODUCTS)
check("gold_top_products has 50 rows",
      products_df.count() == 50,
      f"{products_df.count()} products")

check("gold_top_products rank starts at 1",
      products_df.filter(col("revenue_rank") == 1).count() == 1,
      "rank 1 exists")

# Monthly trend
monthly_df = spark.read.table(GOLD_MONTHLY)
check("gold_monthly_trend has rows",
      monthly_df.count() > 0,
      f"{monthly_df.count()} month rows")

# Customer RFM
rfm_df = spark.read.table(GOLD_RFM)
check("gold_customer_rfm has rows",
      rfm_df.count() > 0,
      f"{rfm_df.count()} customers")

check("RFM recency_days all >= 0",
      rfm_df.filter(col("recency_days") < 0).count() == 0,
      "no negative recency")

check("RFM monetary all > 0",
      rfm_df.filter(col("monetary") <= 0).count() == 0,
      "all monetary values positive")

✅ PASS | gold_revenue_by_country has rows | 41 countries
✅ PASS | gold_revenue_by_country has UK | UK is top market in dataset
✅ PASS | gold_top_products has 50 rows | 50 products
✅ PASS | gold_top_products rank starts at 1 | rank 1 exists
✅ PASS | gold_monthly_trend has rows | 25 month rows
✅ PASS | gold_customer_rfm has rows | 5878 customers
✅ PASS | RFM recency_days all >= 0 | no negative recency
✅ PASS | RFM monetary all > 0 | all monetary values positive


#### ── Final Summary Report

In [0]:
from pyspark.sql import Row

passed = [r for r in results if "PASS" in r["status"]]
failed = [r for r in results if "FAIL" in r["status"]]

print("\n" + "=" * 60)
print(f"  SUMMARY: {len(passed)} passed / {len(failed)} failed / {len(results)} total")
print("=" * 60)

if failed:
    print("\nFailed checks:")
    for r in failed:
        print(f"  ❌ {r['check']} — {r['detail']}")
else:
    print("\n  All checks passed. Pipeline is healthy.")

# Create a summary table you can query in Databricks SQL
summary_df = spark.createDataFrame([
    Row(check=r["check"], status=r["status"], detail=r["detail"])
    for r in results
])

summary_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.new_project.data_quality_report")

print("\nQuality report saved → workspace.new_project.data_quality_report")


  SUMMARY: 27 passed / 0 failed / 27 total

  All checks passed. Pipeline is healthy.

Quality report saved → workspace.new_project.data_quality_report


In [0]:
# See all results as a table in Databricks SQL
display(spark.read.table("workspace.new_project.data_quality_report"))

check,status,detail
Silver < Bronze,✅ PASS,"768,862 < 1,067,371"
Silver retains >= 60%,✅ PASS,Removed 28.0% of rows
Silver has 100K+ rows,✅ PASS,"768,862 rows"
No nulls in invoice_id,✅ PASS,0 nulls found
No nulls in customer_id,✅ PASS,0 nulls found
No nulls in stock_code,✅ PASS,0 nulls found
No nulls in quantity,✅ PASS,0 nulls found
No nulls in unit_price,✅ PASS,0 nulls found
No nulls in total_amount,✅ PASS,0 nulls found
No duplicate invoice+stock rows,✅ PASS,0 duplicate rows found
